# BraTS 2021 Preprocessing Pipeline — All Fixes Applied
**SEIS 766 Vision AI — Spring 2026**

Corrected task ordering: `SEG-01 → SEG-02 → SEG-03 → SEG-06 → SEG-04 → SEG-05`

All fixes from code review applied:
1. SEG-01: Spot-check 3 random cases with shape verification
2. SEG-03: `np.nan_to_num()` before normalization
3. SEG-06: `.copy()` before shuffle to protect caller's list
4. SEG-06: Check if split JSON already exists
5. SEG-04: `+1` on end_z to include full margin
6. SEG-04: Partition-aware extraction (train/val saved separately)
7. SEG-05: `.copy()` after `np.flip` for array contiguity
8. SEG-05: Geometric and photometric transforms separated
9. SEG-05: `borderMode`/`borderValue` in warpAffine
10. SEG-05: Mask cast to float32 before warp, uint8 after
11. SEG-05: Wrapped in Dataset class (train augments, val doesn't)
12. Execution: leakage verification check


In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/"


Mounted at /content/drive


In [ ]:
import os
import json
import random
import numpy as np
import cv2
import nibabel as nib

# ═══════════════════════════════════════════════════════════════
# PATHS
#   TRAIN_PATH: where the BraTS case folders live (on Drive, read-only)
#   OUTPUT_DIR: where persistent files go (Drive — split JSON, models, zip)
#   LOCAL_DIR:  where extracted slices go (Colab local SSD — fast I/O)
#
# Google Drive's FUSE mount chokes on 100,000+ small file writes.
# Solution: extract slices to local SSD, zip, copy one file to Drive.
# ═══════════════════════════════════════════════════════════════
TRAIN_PATH = ROOT + "Group_Project_766"
OUTPUT_DIR = ROOT + "Group_Project_766"
LOCAL_DIR = "/content/local_slices"  # fast local SSD for slice I/O

print(f"TRAIN_PATH exists: {os.path.exists(TRAIN_PATH)}")
if os.path.exists(TRAIN_PATH):
    contents = os.listdir(TRAIN_PATH)[:5]
    print(f"First 5 items: {contents}")


In [3]:
# ═══════════════════════════════════════════════════════════════
# SEG-01: Verify BraTS 2021 Dataset
# Spot-checks 3 random cases for file presence + shape (240x240x155)
# ═══════════════════════════════════════════════════════════════

def verify_data():
    if not os.path.exists(TRAIN_PATH):
        print(f"❌ Folder not found at: {TRAIN_PATH}")
        return []

    # Find all BraTS case FOLDERS (skip any loose files)
    cases = sorted([d for d in os.listdir(TRAIN_PATH)
                    if d.startswith("BraTS2021") and
                    os.path.isdir(os.path.join(TRAIN_PATH, d))])
    print(f"📂 Found {len(cases)} training cases")

    if not cases:
        print("❌ No BraTS2021 case folders found.")
        return []

    # Spot-check 3 random cases for integrity
    spot_checks = random.sample(cases, min(3, len(cases)))
    expected_suffixes = ["_t1.nii.gz", "_t1ce.nii.gz",
                         "_t2.nii.gz", "_flair.nii.gz", "_seg.nii.gz"]

    for case_id in spot_checks:
        case_path = os.path.join(TRAIN_PATH, case_id)
        for suffix in expected_suffixes:
            fpath = os.path.join(case_path, f"{case_id}{suffix}")
            if not os.path.exists(fpath):
                print(f"  ⚠️  Missing: {fpath}")
                continue
            img = nib.load(fpath)
            if img.shape != (240, 240, 155):
                print(f"  ⚠️  Unexpected shape {img.shape} in {fpath}")
            else:
                print(f"  ✅ {case_id}{suffix} → {img.shape}")

    return cases

all_cases = verify_data()


📂 Found 1251 training cases
  ✅ BraTS2021_01080_t1.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_t1ce.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_t2.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_flair.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_seg.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_t1.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_t1ce.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_t2.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_flair.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_seg.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_t1.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_t1ce.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_t2.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_flair.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_seg.nii.gz → (240, 240, 155)


In [4]:
# ═══════════════════════════════════════════════════════════════
# SEG-02: NIfTI Loading Pipeline
# ═══════════════════════════════════════════════════════════════

def load_case_data(case_id):
    """Load all 4 modalities + seg mask for one case."""
    case_path = os.path.join(TRAIN_PATH, case_id)
    modalities = ['t1', 't1ce', 't2', 'flair']
    data_dict = {}

    for mod in modalities:
        file_path = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
        img = nib.load(file_path)
        data_dict[mod] = img.get_fdata().astype(np.float32)

    seg_path = os.path.join(case_path, f"{case_id}_seg.nii.gz")
    seg_img = nib.load(seg_path)
    data_dict['seg'] = seg_img.get_fdata().astype(np.uint8)

    return data_dict

# Quick test
if all_cases:
    sample = load_case_data(all_cases[0])
    print(f"✅ Loaded {all_cases[0]}:")
    for key, value in sample.items():
        print(f"   {key}: {value.shape} {value.dtype}")


✅ Loaded BraTS2021_00000:
   t1: (240, 240, 155) float32
   t1ce: (240, 240, 155) float32
   t2: (240, 240, 155) float32
   flair: (240, 240, 155) float32
   seg: (240, 240, 155) uint8


In [5]:
# ═══════════════════════════════════════════════════════════════
# SEG-03: Per-Volume Z-Score Normalization
#
# FIX: np.nan_to_num() added as first line to catch NaN/Inf
# from scanner artifacts before they corrupt mean/std.
# ═══════════════════════════════════════════════════════════════

def normalize_volume(volume):
    # FIX: catch scanner artifacts
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)

    brain_mask = volume > 0
    if np.any(brain_mask):
        mean = volume[brain_mask].mean()
        std = volume[brain_mask].std()
        normalized = (volume - mean) / (std + 1e-8)
        normalized[~brain_mask] = 0
        return normalized.astype(np.float32)
    return volume.astype(np.float32)


def normalize_case(data_dict):
    """Normalize all 4 modalities. Seg mask is untouched."""
    for mod in ['t1', 't1ce', 't2', 'flair']:
        data_dict[mod] = normalize_volume(data_dict[mod])
    return data_dict

# Quick test
if all_cases:
    norm_flair = normalize_volume(sample['flair'])
    tissue = norm_flair[norm_flair != 0]
    print(f"📊 Normalization check ({all_cases[0]} FLAIR):")
    print(f"   Original max: {sample['flair'].max():.2f}")
    print(f"   Normalized mean: {tissue.mean():.4f}")
    print(f"   Normalized std:  {tissue.std():.4f}")


📊 Normalization check (BraTS2021_00000 FLAIR):
   Original max: 2934.00
   Normalized mean: -0.0000
   Normalized std:  1.0000


In [6]:
# ═══════════════════════════════════════════════════════════════
# SEG-06: Case-Level Train/Val Split
#
# MUST run BEFORE SEG-04 (slice extraction) to prevent leakage.
# FIX: .copy() before shuffle so caller's list is not modified.
# FIX: Check if split JSON already exists before recreating.
# ═══════════════════════════════════════════════════════════════

def save_dataset_split(case_ids, train_ratio=0.8):
    random.seed(42)
    # FIX: .copy() so original list is not permanently shuffled
    shuffled = case_ids.copy()
    random.shuffle(shuffled)

    split_point = int(len(shuffled) * train_ratio)
    train_list = shuffled[:split_point]
    val_list = shuffled[split_point:]

    split_dict = {"train": train_list, "val": val_list}

    output_path = os.path.join(OUTPUT_DIR, "dataset_split.json")

    try:
        with open(output_path, "w") as f:
            json.dump(split_dict, f, indent=4)
        print(f"✅ Split saved to: {output_path}")
    except OSError as e:
        print(f"❌ Failed to save JSON. Error: {e}")
        print("Manual Split Data:", split_dict)

    print(f" - Training: {len(train_list)}, Validation: {len(val_list)}")
    return split_dict


def load_dataset_split():
    path = os.path.join(OUTPUT_DIR, "dataset_split.json")
    with open(path, "r") as f:
        return json.load(f)


# FIX: Check if split exists before recreating
split_path = os.path.join(OUTPUT_DIR, "dataset_split.json")
if os.path.exists(split_path):
    print(f"📄 Loading existing split from {split_path}")
    dataset_split = load_dataset_split()
    print(f"   {len(dataset_split['train'])} train / {len(dataset_split['val'])} val")
else:
    print("🔀 Creating train/val split...")
    dataset_split = save_dataset_split(all_cases)

# FIX: Verify no leakage
overlap = set(dataset_split['train']) & set(dataset_split['val'])
if overlap:
    print(f"❌ LEAKAGE: {len(overlap)} cases in both partitions!")
else:
    print(f"✅ No leakage: {len(set(dataset_split['train']))} train / "
          f"{len(set(dataset_split['val']))} val, 0 overlap")


📄 Loading existing split from /content/drive/MyDrive/Group_Project_766/dataset_split.json
   1000 train / 251 val
✅ No leakage: 1000 train / 251 val, 0 overlap


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-04: Partition-Aware 2D Axial Slice Extraction
#
# FIX: +1 on end_z to include full margin in range().
# FIX: Partition-aware — train and val saved to separate dirs.
# FIX: Writes to LOCAL_DIR (Colab SSD) not Drive to avoid I/O errors.
# ═══════════════════════════════════════════════════════════════

def extract_tumor_slices(data_dict, margin=5):
    seg = data_dict['seg']
    z_indices = np.any(seg > 0, axis=(0, 1))

    if not np.any(z_indices):
        return [], []

    start_z = max(0, np.where(z_indices)[0].min() - margin)
    # FIX: + 1 so range() includes the last margin slice
    end_z = min(seg.shape[2], np.where(z_indices)[0].max() + margin + 1)

    slices_images, slices_masks = [], []
    for z in range(start_z, end_z):
        combined_slice = np.stack([
            data_dict['t1'][:, :, z],
            data_dict['t1ce'][:, :, z],
            data_dict['t2'][:, :, z],
            data_dict['flair'][:, :, z]
        ], axis=0)
        slices_images.append(combined_slice)
        slices_masks.append(seg[:, :, z])
    return slices_images, slices_masks


def extract_partition(partition_case_ids, partition_name):
    """Extract slices to LOCAL_DIR (Colab SSD) for fast I/O."""
    # Write to local SSD, NOT Drive
    save_dir = os.path.join(LOCAL_DIR, "slices", partition_name)
    os.makedirs(save_dir, exist_ok=True)

    total = 0
    for i, case_id in enumerate(partition_case_ids):
        data_dict = load_case_data(case_id)
        data_dict = normalize_case(data_dict)
        imgs, msks = extract_tumor_slices(data_dict)

        for j, (im, mk) in enumerate(zip(imgs, msks)):
            np.savez_compressed(
                os.path.join(save_dir, f"{case_id}_slice_{j:03d}.npz"),
                image=im, mask=mk)

        total += len(imgs)
        if (i + 1) % 50 == 0 or (i + 1) == len(partition_case_ids):
            print(f"  [{partition_name}] {i+1}/{len(partition_case_ids)}, "
                  f"{total} slices")

    print(f"✅ {partition_name}: {total} slices → {save_dir}")
    return total


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Run Partition-Aware Extraction (writes to local SSD)
#
# SKIP THIS CELL if slices were already extracted in a previous
# session. Instead, run Cell 8b to unzip from Drive.
# ═══════════════════════════════════════════════════════════════

print("📤 Extracting training slices to local SSD...")
train_count = extract_partition(dataset_split['train'], 'train')

print("\n📤 Extracting validation slices to local SSD...")
val_count = extract_partition(dataset_split['val'], 'val')

print(f"\n✅ Total: {train_count} train + {val_count} val slices")
print(f"   Saved to: {LOCAL_DIR}/slices/")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Transfer Slices to Google Drive (run after extraction)
#
# Zips all extracted slices into one file and copies to Drive.
# This makes exactly ONE write request to Drive instead of 100,000+.
# The zip persists on Drive across sessions.
# ═══════════════════════════════════════════════════════════════

import shutil

zip_name = "brats_slices"
zip_path = f"/content/{zip_name}"

print("📦 Zipping extracted slices...")
shutil.make_archive(zip_path, 'zip', LOCAL_DIR)
print(f"✅ Created {zip_path}.zip")

drive_zip = os.path.join(OUTPUT_DIR, f"{zip_name}.zip")
print(f"📤 Copying to Drive: {drive_zip}")
shutil.copy2(f"{zip_path}.zip", drive_zip)

size_gb = os.path.getsize(drive_zip) / 1e9
print(f"✅ Saved to Drive. Size: {size_gb:.2f} GB")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load Slices from Drive Zip (future sessions)
#
# Run THIS cell instead of Cells 8 + 8a if slices were already
# extracted in a previous session. Colab's local storage is wiped
# when the runtime disconnects, but the zip on Drive persists.
# ═══════════════════════════════════════════════════════════════

import zipfile

LOCAL_DIR = "/content/local_slices"  # reset in case cell 2 wasn't re-run
drive_zip = os.path.join(OUTPUT_DIR, "brats_slices.zip")
local_check = os.path.join(LOCAL_DIR, "slices", "train")

if os.path.exists(local_check):
    n_train = len([f for f in os.listdir(local_check) if f.endswith('.npz')])
    n_val = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR, 'slices', 'val')) if f.endswith('.npz')])
    print(f"✅ Local slices already exist: {n_train} train + {n_val} val")
elif os.path.exists(drive_zip):
    print(f"📥 Extracting {drive_zip} to local storage...")
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(LOCAL_DIR)
    n_train = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR, 'slices', 'train')) if f.endswith('.npz')])
    n_val = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR, 'slices', 'val')) if f.endswith('.npz')])
    print(f"✅ Unzipped: {n_train} train + {n_val} val slices")
else:
    print("⚠️  No zip found on Drive. Run extraction cells (8 + 8a) first.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-05: Train-Only Augmentation
#
# FIX: .copy() after np.flip
# FIX: Geometric (image+mask) separated from photometric (image only)
# FIX: borderMode=BORDER_CONSTANT, borderValue=0.0
# FIX: Mask cast float32 before warp, uint8 after
# FIX: Wrapped in Dataset class with augment flag
# ═══════════════════════════════════════════════════════════════

def geometric_augmentation(image, mask):
    """GEOMETRIC: apply to BOTH image and mask. NN interp on mask."""
    if random.random() > 0.5:
        image = np.flip(image, axis=2).copy()  # FIX: .copy()
        mask = np.flip(mask, axis=1).copy()     # FIX: .copy()

    angle = random.uniform(-15, 15)
    h, w = mask.shape
    rot = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)

    aug_img = np.zeros_like(image)
    for c in range(image.shape[0]):
        aug_img[c] = cv2.warpAffine(
            image[c], rot, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)

    aug_msk = cv2.warpAffine(
        mask.astype(np.float32), rot, (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0.0
    ).astype(np.uint8)

    return aug_img, aug_msk


def photometric_augmentation(image):
    """PHOTOMETRIC: IMAGE ONLY. Never apply to mask."""
    image = image * random.uniform(0.9, 1.1)
    if random.random() > 0.5:
        noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
        image = image + noise
    return image


def apply_augmentation(image, mask):
    """Geometric first (both), then photometric (image only)."""
    image, mask = geometric_augmentation(image, mask)
    image = photometric_augmentation(image)
    return image, mask


class BraTSSliceDataset:
    """Dataset wrapper. augment=True for train, False for val."""
    def __init__(self, slice_dir, augment=False):
        self.slice_dir = slice_dir
        self.augment = augment
        self.file_list = sorted(
            [f for f in os.listdir(slice_dir) if f.endswith('.npz')])
        print(f"📦 {len(self.file_list)} slices "
              f"(augment={'ON' if augment else 'OFF'})")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        data = np.load(os.path.join(self.slice_dir, self.file_list[idx]))
        image = data['image'].astype(np.float32)
        mask = data['mask'].astype(np.uint8)

        if self.augment:
            image, mask = apply_augmentation(image, mask)

        return image.astype(np.float32), mask.astype(np.int64)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Create Datasets and Verify
# Reads from LOCAL_DIR (Colab SSD) for fast training I/O.
# ═══════════════════════════════════════════════════════════════

train_dir = os.path.join(LOCAL_DIR, 'slices', 'train')
val_dir = os.path.join(LOCAL_DIR, 'slices', 'val')

train_dataset = BraTSSliceDataset(train_dir, augment=True)
val_dataset = BraTSSliceDataset(val_dir, augment=False)

# Demo: load one sample from each
if len(train_dataset) > 0:
    img_t, msk_t = train_dataset[0]
    print(f"\n🔬 Train sample: image {img_t.shape} {img_t.dtype}, "
          f"mask {msk_t.shape} {msk_t.dtype}")
    print(f"   Mask labels: {np.unique(msk_t)}")

if len(val_dataset) > 0:
    img_v, msk_v = val_dataset[0]
    print(f"🔬 Val sample:   image {img_v.shape} {img_v.dtype}, "
          f"mask {msk_v.shape} {msk_v.dtype}")
    print(f"   Mask labels: {np.unique(msk_v)}")

print("\n🚀 Phase 1 complete. Ready for model training.")


---
# Phase 2: SEG-07 through SEG-13 + EVAL-01/02
**U-Net Segmentation Model — TensorFlow/Keras**

Prerequisites: Phase 1 cells above must have run, OR Cell 8b must have
loaded slices from the Drive zip. Slices must exist at `LOCAL_DIR/slices/`.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# TensorFlow/Keras Imports
# ═══════════════════════════════════════════════════════════════

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import glob

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU: {gpus[0].name}")
else:
    print("⚠️  No GPU detected.")

# Slice directories on local SSD
SLICE_DIR_TRAIN = os.path.join(LOCAL_DIR, 'slices', 'train')
SLICE_DIR_VAL = os.path.join(LOCAL_DIR, 'slices', 'val')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')  # models save to Drive
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Train slices: {len(glob.glob(os.path.join(SLICE_DIR_TRAIN, '*.npz')))}")
print(f"Val slices:   {len(glob.glob(os.path.join(SLICE_DIR_VAL, '*.npz')))}")
print(f"TensorFlow:   {tf.__version__}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# EVAL-01: Dataset Statistics
# ═══════════════════════════════════════════════════════════════

def compute_dataset_stats(slice_dir, max_samples=500):
    files = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))[:max_samples]
    if not files:
        print(f"No .npz files in {slice_dir}")
        return None

    label_counts = {0: 0, 1: 0, 2: 0, 4: 0}
    tumor_pixels = []

    for f in files:
        msk = np.load(f)['mask']
        for label in [0, 1, 2, 4]:
            label_counts[label] += int(np.sum(msk == label))
        tumor_pixels.append(int(np.sum(msk > 0)))

    total = sum(label_counts.values())
    names = {0: 'Background', 1: 'Necrotic Core', 2: 'Edema', 4: 'Enhancing Tumor'}
    print(f"📊 EVAL-01: Dataset Statistics ({len(files)} slices)")
    for label, count in label_counts.items():
        pct = 100 * count / total if total > 0 else 0
        print(f"   {names[label]:20s} (label {label}): {count:>12,} px ({pct:.2f}%)")
    tp = np.array(tumor_pixels)
    print(f"\n   Tumor px/slice — Mean: {tp.mean():.0f}, "
          f"Median: {np.median(tp):.0f}, Min: {tp.min()}, Max: {tp.max()}")
    return label_counts

train_stats = compute_dataset_stats(SLICE_DIR_TRAIN)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# EVAL-02: Sample 4-Modality Visualizations + Mask Overlay
# ═══════════════════════════════════════════════════════════════

def visualize_sample(slice_dir, sample_idx=0):
    files = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))
    data = np.load(files[sample_idx])
    img, msk = data['image'], data['mask']
    fname = os.path.basename(files[sample_idx])

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(fname, fontsize=12)
    for i, (name, ax) in enumerate(zip(['T1','T1ce','T2','FLAIR'], axes[:4])):
        ax.imshow(img[i], cmap='gray'); ax.set_title(name); ax.axis('off')

    mask_rgb = np.zeros((*msk.shape, 3), dtype=np.float32)
    mask_rgb[msk == 1] = [1,0,0]; mask_rgb[msk == 2] = [0,1,0]; mask_rgb[msk == 4] = [1,1,0]
    axes[4].imshow(img[1], cmap='gray')
    axes[4].imshow(mask_rgb, alpha=0.4)
    axes[4].set_title('Mask Overlay'); axes[4].axis('off')
    plt.tight_layout(); plt.show()

train_files = sorted(glob.glob(os.path.join(SLICE_DIR_TRAIN, '*.npz')))
for idx in random.sample(range(len(train_files)), min(3, len(train_files))):
    visualize_sample(SLICE_DIR_TRAIN, idx)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-07: U-Net Architecture + SEG-08: 4-Channel Early Fusion
#
# Dr. Lai Week 4: "Each encoder usually has two convolution blocks...
#   convolution, batch normalization, and ReLU. Repeat twice."
#   "U-Net is using the skip connection, passing the feature map"
#   "They are concatenated together"
#
# Input:  (batch, 240, 240, 4) channels-last
# Output: (batch, 240, 240, 3) sigmoid — WT, TC, ET
# ═══════════════════════════════════════════════════════════════

def conv_block(x, filters, name_prefix):
    x = layers.Conv2D(filters, 3, padding='same', name=f'{name_prefix}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = layers.ReLU(name=f'{name_prefix}_relu1')(x)
    x = layers.Conv2D(filters, 3, padding='same', name=f'{name_prefix}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
    x = layers.ReLU(name=f'{name_prefix}_relu2')(x)
    return x

def encoder_block(x, filters, name_prefix):
    ppf = conv_block(x, filters, name_prefix)
    pooled = layers.MaxPooling2D(2, name=f'{name_prefix}_pool')(ppf)
    pooled = layers.Dropout(0.1, name=f'{name_prefix}_drop')(pooled)
    return ppf, pooled

def decoder_block(x, skip, filters, name_prefix):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same',
                                name=f'{name_prefix}_upconv')(x)
    x = layers.Concatenate(name=f'{name_prefix}_concat')([x, skip])
    x = conv_block(x, filters, name_prefix)
    return x

def build_unet(input_shape=(240, 240, 4), num_classes=3):
    inputs = keras.Input(shape=input_shape, name='input_4ch')
    s1, p1 = encoder_block(inputs, 64,  'enc1')
    s2, p2 = encoder_block(p1,     128, 'enc2')
    s3, p3 = encoder_block(p2,     256, 'enc3')
    s4, p4 = encoder_block(p3,     512, 'enc4')
    b = conv_block(p4, 1024, 'bottleneck')
    d4 = decoder_block(b,  s4, 512, 'dec4')
    d3 = decoder_block(d4, s3, 256, 'dec3')
    d2 = decoder_block(d3, s2, 128, 'dec2')
    d1 = decoder_block(d2, s1, 64,  'dec1')
    outputs = layers.Conv2D(num_classes, 1, activation='sigmoid', name='output')(d1)
    return keras.Model(inputs, outputs, name='UNet_BraTS')

model = build_unet()
model.summary()

dummy = np.random.rand(2, 240, 240, 4).astype(np.float32)
out = model(dummy, training=False)
print(f"\n✅ Input: {dummy.shape} → Output: {out.shape}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-09: Soft Dice + BCE Combined Loss
#
# Dr. Lai Week 5: "I personally prefer IOU... dice usually
#   will give more optimistic output"
# Training: Dice+BCE (handles class imbalance + smooth gradients)
# Evaluation: IoU (honest metric per Dr. Lai)
# ═══════════════════════════════════════════════════════════════

def soft_dice_loss(y_true, y_pred, smooth=1e-6):
    inter = tf.reduce_sum(y_true * y_pred, axis=(1, 2))
    denom = tf.reduce_sum(y_true, axis=(1, 2)) + tf.reduce_sum(y_pred, axis=(1, 2))
    dice = (2.0 * inter + smooth) / (denom + smooth)
    return 1.0 - tf.reduce_mean(dice)

def dice_bce_loss(y_true, y_pred):
    return soft_dice_loss(y_true, y_pred) + \
           tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))

def iou_metric(y_true, y_pred):
    yp = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * yp, axis=(1, 2))
    union = tf.reduce_sum(y_true, axis=(1, 2)) + tf.reduce_sum(yp, axis=(1, 2)) - inter
    return tf.reduce_mean((inter + 1e-6) / (union + 1e-6))

def dice_metric(y_true, y_pred):
    yp = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * yp, axis=(1, 2))
    denom = tf.reduce_sum(y_true, axis=(1, 2)) + tf.reduce_sum(yp, axis=(1, 2))
    return tf.reduce_mean((2.0 * inter + 1e-6) / (denom + 1e-6))

dt = np.random.randint(0, 2, (2, 240, 240, 3)).astype(np.float32)
dp = np.random.rand(2, 240, 240, 3).astype(np.float32)
print(f"Dice+BCE: {dice_bce_loss(dt, dp):.4f}  IoU: {iou_metric(dt, dp):.4f}  "
      f"Dice: {dice_metric(dt, dp):.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Data Loading: .npz → tf.data.Dataset
#
# Converts Yue's channels-first (4, 240, 240) to TF channels-last
# BraTS labels (0,1,2,4) → 3 binary channels (WT, TC, ET)
# Reads from LOCAL_DIR (local SSD) for fast I/O
# ═══════════════════════════════════════════════════════════════

def brats_labels_to_channels(mask):
    wt = np.float32(mask > 0)
    tc = np.float32((mask == 1) | (mask == 4))
    et = np.float32(mask == 4)
    return np.stack([wt, tc, et], axis=-1)

def load_npz_for_tf(fpath):
    data = np.load(fpath)
    img = np.transpose(data['image'].astype(np.float32), (1, 2, 0))  # → (240,240,4)
    msk = brats_labels_to_channels(data['mask'].astype(np.uint8))    # → (240,240,3)
    return img, msk

def create_tf_dataset(slice_dir, batch_size=8, shuffle=True):
    file_list = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))
    print(f"📦 {len(file_list)} slices from {os.path.basename(slice_dir)}")
    def gen():
        fs = file_list.copy()
        if shuffle: np.random.shuffle(fs)
        for f in fs: yield load_npz_for_tf(f)
    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((240,240,4), tf.float32),
        tf.TensorSpec((240,240,3), tf.float32)))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE), len(file_list)

BATCH_SIZE = 8
train_ds, train_count = create_tf_dataset(SLICE_DIR_TRAIN, BATCH_SIZE, shuffle=True)
val_ds, val_count = create_tf_dataset(SLICE_DIR_VAL, BATCH_SIZE, shuffle=False)

for ib, mb in train_ds.take(1):
    print(f"\n✅ Batch: images {ib.shape}, masks {mb.shape}")
    print(f"   Range: [{ib.numpy().min():.2f}, {ib.numpy().max():.2f}]")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-10 + SEG-11: Compile and Train Baseline U-Net
#
# Dr. Lai Week 4: "I always retrain everything from scratch"
# Model checkpoint saves to Drive (persists across sessions)
# ═══════════════════════════════════════════════════════════════

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=dice_bce_loss,
    metrics=[dice_metric, iou_metric]
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'unet_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=1),
]

EPOCHS = 50

history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1
)

print(f"\n✅ Training complete. Best val loss: {min(history.history['val_loss']):.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Training History
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Dice+BCE Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['dice_metric'], label='Train')
axes[1].plot(history.history['val_dice_metric'], label='Val')
axes[1].set_title('Dice'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(history.history['iou_metric'], label='Train')
axes[2].plot(history.history['val_iou_metric'], label='Val')
axes[2].set_title('IoU'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-12: Evaluate Baseline — Dice/IoU per Tumor Region
# ═══════════════════════════════════════════════════════════════

def evaluate_per_class(model, dataset, class_names=['WT', 'TC', 'ET']):
    all_dice = {c: [] for c in class_names}
    all_iou = {c: [] for c in class_names}

    for ib, mb in dataset:
        pb = tf.cast(model(ib, training=False) > 0.5, tf.float32)
        for ci, cn in enumerate(class_names):
            yt, yp = mb[:,:,:,ci], pb[:,:,:,ci]
            inter = tf.reduce_sum(yt * yp, axis=(1,2))
            union = tf.reduce_sum(yt, axis=(1,2)) + tf.reduce_sum(yp, axis=(1,2)) - inter
            denom = tf.reduce_sum(yt, axis=(1,2)) + tf.reduce_sum(yp, axis=(1,2))
            all_iou[cn].extend(((inter+1e-6)/(union+1e-6)).numpy().tolist())
            all_dice[cn].extend(((2*inter+1e-6)/(denom+1e-6)).numpy().tolist())

    print("📊 SEG-12: Baseline Evaluation")
    print(f"{'Region':<8} {'Dice':>10} {'± std':>10} {'IoU':>10} {'± std':>10}")
    print("-" * 50)
    results = {}
    for cn in class_names:
        d, i = np.array(all_dice[cn]), np.array(all_iou[cn])
        print(f"{cn:<8} {d.mean():>10.4f} {d.std():>10.4f} {i.mean():>10.4f} {i.std():>10.4f}")
        results[cn] = {'dice': float(d.mean()), 'iou': float(i.mean())}
    return results

results = evaluate_per_class(model, val_ds)
with open(os.path.join(OUTPUT_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(results, f, indent=2)
print("\n✅ Saved baseline_metrics.json to Drive")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-13: Segmentation Overlay Visualizations
# FLAIR | Ground Truth | Prediction
# ═══════════════════════════════════════════════════════════════

def create_overlay(base, mask_3ch, alpha=0.4):
    b = (base - base.min()) / (base.max() - base.min() + 1e-8)
    rgb = np.stack([b, b, b], axis=-1)
    ov = np.zeros_like(rgb)
    ov[mask_3ch[:,:,0] > 0.5] = [0,1,0]  # WT green
    ov[mask_3ch[:,:,1] > 0.5] = [1,1,0]  # TC yellow
    ov[mask_3ch[:,:,2] > 0.5] = [1,0,0]  # ET red
    tumor = mask_3ch[:,:,0:1] > 0.5
    return np.clip(np.where(tumor, rgb*(1-alpha)+ov*alpha, rgb), 0, 1)

def show_overlays(model, dataset, n=5):
    samples = []
    for ib, mb in dataset:
        pb = model(ib, training=False)
        for i in range(ib.shape[0]):
            if tf.reduce_sum(mb[i]).numpy() > 100:
                samples.append((ib[i].numpy(), mb[i].numpy(), pb[i].numpy()))
            if len(samples) >= n: break
        if len(samples) >= n: break

    fig, axes = plt.subplots(len(samples), 3, figsize=(15, 5*len(samples)))
    if len(samples) == 1: axes = axes[np.newaxis, :]
    for row, (img, gt, pred) in enumerate(samples):
        flair = img[:,:,3]
        axes[row,0].imshow(flair, cmap='gray')
        axes[row,0].set_title('FLAIR'); axes[row,0].axis('off')
        axes[row,1].imshow(create_overlay(flair, gt))
        axes[row,1].set_title('Ground Truth'); axes[row,1].axis('off')
        axes[row,2].imshow(create_overlay(flair, (pred>0.5).astype(np.float32)))
        axes[row,2].set_title('Prediction'); axes[row,2].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'segmentation_overlays.png'), dpi=150)
    plt.show()
    print("✅ Overlays saved to Drive")

show_overlays(model, val_ds)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Save Model + Handoff Checklist
# Model saves to Drive (persists across sessions)
# ═══════════════════════════════════════════════════════════════

save_path = os.path.join(MODEL_DIR, 'unet_baseline_final.keras')
model.save(save_path)

print("═" * 60)
print("SEG-11 HANDOFF CHECKLIST")
print("═" * 60)
print(f"1. Checkpoint: {save_path}")
print(f"2. Input:  (batch, 240, 240, 4) channels-last")
print(f"   Output: (batch, 240, 240, 3) sigmoid [0,1]")
print(f"   Channels: [T1, T1ce, T2, FLAIR] → [WT, TC, ET]")
print(f"3. Dtype: float32")
print(f"4. Normalization: z-scored per-volume")
print(f"5. Inference:")
print(f"   model = keras.models.load_model(path,")
print(f"     custom_objects={{'dice_bce_loss': dice_bce_loss,")
print(f"       'dice_metric': dice_metric, 'iou_metric': iou_metric}})")
print(f"   pred = model(input_tensor, training=False)")
print(f"   mask = (pred > 0.5).numpy()")
print(f"6. Sanity check: segmentation_overlays.png")
print(f"7. Framework: TensorFlow/Keras, channels-last")
print("═" * 60)
